In [142]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [143]:
# Change these to your paths.
TRAIN_CSV = "dataset/train_data.csv"
TRAIN_IMG_DIR = "dataset/train_imgs"
TEST_CSV = "dataset/test_data.csv"
TEST_IMG_DIR = "dataset/test_imgs"


In [144]:
MODEL_ID = "openai/clip-vit-base-patch16"
clip = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
proc = CLIPProcessor.from_pretrained(MODEL_ID, use_fast=True)

def clip_feature_tensor(features):
    if torch.is_tensor(features):
        return features
    if hasattr(features, "pooler_output") and features.pooler_output is not None:
        return features.pooler_output
    if isinstance(features, (tuple, list)):
        return features[0]
    raise TypeError(f"Unexpected CLIP feature output: {type(features)!r}")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 51999.28it/s]


In [145]:
train = pd.read_csv(TRAIN_CSV)  # card-photo pairs; not used by this baseline
test = pd.read_csv(TEST_CSV)
gal_paths = sorted(Path(TEST_IMG_DIR).glob("*.jpg"))
gal_ids = np.array([p.stem for p in gal_paths])
print(f"train pairs: {len(train)}, queries: {len(test)}, gallery: {len(gal_paths)}")
print(test.iloc[0]["description"][:200])


train pairs: 500, queries: 700, gallery: 1500
Stone: A beige stone on the front of the short wall on the bottom right corner of the image.
Stone: A natural grey stone on the stairs.
Stone: A beige stone on the front of the short wall on the bot


In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class TrainPairs(Dataset):
    def __init__(self, df):
        self.df = df
        super().__init__()

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(f"{TRAIN_IMG_DIR}/{row.image_id}.jpg")
        text = row.description
        return image, text, row.image_id
    
def collate(batch):
    images, text, image_ids = zip(*batch)
    encoded = proc(
        text=list(text),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    encoded["image_ids"] = list(image_ids)
    return encoded

rows = []
chunk_size = 5

for _, row in train.iterrows():
    image_id = row["image_id"]
    text = str(row["description"])

    rows.append({"image_id": image_id, "description": text})

    linetext = []
    for line in text.splitlines():
        line = line.strip()
        if len(line) <= 5:
            continue

        linetext.append(line)
        if len(linetext) == chunk_size:
            rows.append({"image_id": image_id, "description": " ".join(linetext)})
            linetext = []

    if linetext:
        rows.append({"image_id": image_id, "description": " ".join(linetext)})

mixed_df = pd.DataFrame(rows)
print(f"training rows after blob chunking: {len(train)} -> {len(mixed_df)}")

train_df, val_df = train_test_split(mixed_df, test_size=0.1, random_state=42)

train_loader = DataLoader(TrainPairs(train_df), batch_size=8, collate_fn=collate)
val_loader = DataLoader(TrainPairs(val_df), batch_size=8, collate_fn=collate)

mixed_df.head()

In [ ]:
import torch

def multi_positive_contrastive_loss(logits, positive_mask):
    log_probs = logits - logits.logsumexp(dim=1, keepdim=True)
    positives_per_anchor = positive_mask.sum(dim=1).clamp_min(1)
    return -(log_probs * positive_mask).sum(dim=1).div(positives_per_anchor).mean()


clip.train()

optimizer = torch.optim.AdamW(clip.parameters(), lr=1e-6)

epochs = 2

for epoch in range(epochs):
    total_loss = 0

    for batch in train_loader:
        image_ids = batch.pop("image_ids")
        batch = {k: v.to(device) for k, v in batch.items()}

        out = clip(**batch)

        logits_per_image = out.logits_per_image
        logits_per_text = out.logits_per_text

        image_ids = np.array(image_ids)
        positive_mask = torch.tensor(
            image_ids[:, None] == image_ids[None, :],
            dtype=logits_per_image.dtype,
            device=device,
        )

        loss_i = multi_positive_contrastive_loss(logits_per_image, positive_mask)
        loss_t = multi_positive_contrastive_loss(logits_per_text, positive_mask.T)
        loss = (loss_i + loss_t) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # print(out)
        # print(positive_mask)
        # print(loss_i)

    print (f"Epoch {epoch+1}: total loss: {total_loss / len(train_loader):.4f}")

In [148]:
@torch.inference_mode()
def image_vecs(paths, bs=64):
    out = []
    for i in range(0, len(paths), bs):
        ims = [Image.open(p) for p in paths[i:i + bs]]
        px = proc(images=ims, return_tensors="pt")["pixel_values"].to(device)
        v = clip_feature_tensor(clip.get_image_features(pixel_values=px))
        out.append(F.normalize(v, dim=-1).cpu())
    return torch.cat(out)

gal = image_vecs(gal_paths)

In [149]:
@torch.inference_mode()
def text_vecs(strs, bs=128):
    out = []
    for i in range(0, len(strs), bs):
        inp = proc(text=strs[i:i + bs], return_tensors="pt", padding=True, truncation=True).to(device)
        v = clip_feature_tensor(clip.get_text_features(**inp))
        out.append(F.normalize(v, dim=-1).cpu())
    return torch.cat(out)

In [150]:
def chunk_description(text, chunk_size=5):
    text = str(text)
    chunks = [text]
    linetext = []

    for line in text.splitlines():
        line = line.strip()
        if len(line) <= 5:
            continue

        linetext.append(line)
        if len(linetext) == chunk_size:
            chunks.append(" ".join(linetext))
            linetext = []

    if linetext:
        chunks.append(" ".join(linetext))

    return chunks


rows = []
for _, row in test.iterrows():
    datapoint_id = row["datapointID"]
    for chunk in chunk_description(row["description"], chunk_size=5):
        rows.append({"datapointID": datapoint_id, "description": chunk})

test_chunks = pd.DataFrame(rows)
print(f"test rows after blob chunking: {len(test)} -> {len(test_chunks)}")

chunk_vecs = text_vecs(test_chunks["description"].tolist())
test_chunks["qvec"] = list(chunk_vecs.numpy())

qvec_mean = (
    test_chunks.drop(columns=["description"])
    .groupby("datapointID", sort=False)["qvec"]
    .apply(lambda vecs: np.stack(vecs).mean(axis=0))
)
qvec_mean = qvec_mean.reindex(test["datapointID"].values)
qvec_mean = torch.tensor(np.stack(qvec_mean.to_numpy()), dtype=gal.dtype)
qvec_mean = F.normalize(qvec_mean, dim=-1)

sims = qvec_mean @ gal.T
top = sims.topk(10).indices.numpy()
answers = [" ".join(gal_ids[idx]) for idx in top]

sub = pd.DataFrame({"subtaskID": 1,
                    "datapointID": test["datapointID"].values,
                    "answer": answers})
sub.to_csv("submission.csv", index=False)
sub.head()

test rows after blob chunking: 700 -> 8644


,subtaskID,datapointID,answer
0,1,0,img_00008 img_01045 img_01483 img_00387 img_00...
1,1,1,img_00581 img_00578 img_00382 img_01466 img_01...
2,1,2,img_00844 img_00837 img_00195 img_00168 img_01...
3,1,3,img_00361 img_01430 img_00954 img_00953 img_00...
4,1,4,img_01360 img_01145 img_00120 img_00037 img_00...
